In [4]:
import pandas as pd
import spacy as spc
import scispacy as scspcy
# Multiple ways to get and work with home directory
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")
#files = os.listdir(os.getcwd())
#print(files)

# Step 3: Download and load the model
try:
    nlp = spacy.load("en_core_sci_sm")
    
    # Test the model
    text = "Paracetamol is commonly used to treat fever and mild to moderate pain."
    doc = nlp(text)
    
    # Print results
    print("Named Entities:")
    for ent in doc.ents:
        print(f"Entity: {ent.text}, Label: {ent.label_}")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running this in a fresh Python session")
    print("2. Try restarting your Jupyter kernel")
    print("3. Verify your Python environment has write permissions")



Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi

Contents of home directory:
Current working directory: /Users/naveendeevi/Datasets
Error: name 'spacy' is not defined

Troubleshooting steps:
1. Make sure you're running this in a fresh Python session
2. Try restarting your Jupyter kernel
3. Verify your Python environment has write permissions


In [3]:
## Functions

def standardize_dataframe_sklearn(df):
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(df)
    standardized_df = pd.DataFrame(scaled_values, columns=df.columns, index=df.index)
    return standardized_df, scaler

def unstandardize_dataframe(standardized_df, scaler):
    original_values = scaler.inverse_transform(standardized_df)
    return pd.DataFrame(original_values, columns=standardized_df.columns, index=standardized_df.index)

def drop_missing_columns(df, threshold=0.8):
   missing_percentages = df.isnull().mean()
   
   # Identify columns to drop
   columns_to_drop = missing_percentages[missing_percentages > threshold].index
   
   # Drop the identified columns
   df = df.drop(columns=columns_to_drop)
   return df

nlp = spacy.load("en_ner_bc5cdr_md")

def text_to_disease(drug_name):
    #print(f"Processing: {drug_name}")
    
    # Construct FDA URL
    fda_url = f'https://api.fda.gov/drug/label.json?search=openfda.brand_name:"{drug_name.lower()}"'
    
    try:
        fda_resp = requests.get(fda_url).json()
        results = fda_resp.get('results', [])

        if results and 'indications_and_usage' in results[0]:
            indication_large_body = results[0]['indications_and_usage'][0]
        else:
            return "No indications info found", []
    except Exception as e:
        print(f"Error fetching FDA data for {drug_name}: {e}")
        return "No indications info found", []

    # Clean and process text
    text_clean = ''.join(c.lower() if c.isalpha() else c for c in indication_large_body)
    result = nlp(text_clean)
    result_ents = [str(ent) for ent in result.ents if str(ent).lower() != drug_name.lower()]

    # Count and sort
    counted = Counter(result_ents)
    sorted_result = sorted(counted.items(), key=lambda x: x[1], reverse=True)

    return indication_large_body, sorted_result

# Safe wrapper to ensure consistency
def safe_text_to_disease(drug_name):
    try:
        output = text_to_disease(drug_name)
        if isinstance(output, tuple) and len(output) == 2:
            return output
        else:
            return "No indications info found", []
    except Exception as e:
        print(f"Exception in processing {drug_name}: {e}")
        return "No indications info found", []

NameError: name 'spacy' is not defined

In [4]:
# Basic reading of CSV file
df = pd.read_csv('MDD_Clustering_V3.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 435 entries, 0 to 434
Data columns (total 27 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   patient_id                                  435 non-null    object 
 1   preferred_name                              435 non-null    object 
 2   gender                                      435 non-null    object 
 3   pat_age                                     435 non-null    int64  
 4   practice_name                               435 non-null    object 
 5   provider_name                               435 non-null    object 
 6   mdd_first_dx_date                           435 non-null    object 
 7   mdd_last_dx_date                            435 non-null    object 
 8   total_mdd_episodes                          435 non-null    int64  
 9   SRI_Refills                                 294 non-null    float64
 10  Atypical_Antip

In [5]:
df = df.set_index(['patient_id','preferred_name'])

### Feature Engineering

In [6]:
# Gender
df['Is_Female'] = [1 if i=='PATIENT_GENDER_FEMALE' else 0 for i in df['gender']]
df['Is_Male'] = [1 if i == 'PATIENT_GENDER_MALE' else 0 for i in df['gender']]

In [7]:
# Length of MDDs first to last
df['mdd_first_dx_date'] = pd.to_datetime(df['mdd_first_dx_date'])
df['mdd_first_dx_since'] = (pd.Timestamp.today().normalize() - df['mdd_first_dx_date']).dt.days

# Length of MDDs first to last
df['mdd_last_dx_date'] = pd.to_datetime(df['mdd_last_dx_date'])
df['mdd_latest_dx_since'] = (pd.Timestamp.today().normalize() - df['mdd_last_dx_date']).dt.days

In [8]:
df.columns

Index(['gender', 'pat_age', 'practice_name', 'provider_name',
       'mdd_first_dx_date', 'mdd_last_dx_date', 'total_mdd_episodes',
       'SRI_Refills', 'Atypical_Antipsychotic_Refills',
       'Anti_epileptic_agent_Refills', 'Aminoketone_Refills',
       'Benzodiazepine_Refills', 'Monoamine_Oxidase_Type_B_Inhibitor_Refills',
       'Phenothiazine_Refills', 'SNRI_Refills',
       'Tricyclic_Antidepressant_Refills', 'total_logs', 'avg_energy_score',
       'median_energy_score', 'min_energy_score', 'max_energy_score',
       'avg_pleasantness_score', 'median_pleasantness_score',
       'min_pleasantness_score', 'max_pleasantness_score', 'Is_Female',
       'Is_Male', 'mdd_first_dx_since', 'mdd_latest_dx_since'],
      dtype='object')

In [9]:
df_2 = df[['pat_age', 'Is_Female', 'Is_Male', 'mdd_first_dx_since', 'mdd_latest_dx_since','total_mdd_episodes',
          'SRI_Refills','SNRI_Refills','Atypical_Antipsychotic_Refills','Anti_epileptic_agent_Refills',
          'Aminoketone_Refills', 'Benzodiazepine_Refills', 'Monoamine_Oxidase_Type_B_Inhibitor_Refills',
           'Phenothiazine_Refills', 'Tricyclic_Antidepressant_Refills', 'total_logs', 
           'avg_energy_score', 'median_energy_score','min_energy_score','max_energy_score', 
           'avg_pleasantness_score', 'median_pleasantness_score', 'min_pleasantness_score', 'max_pleasantness_score'
          ]]

In [10]:
df_2.columns

Index(['pat_age', 'Is_Female', 'Is_Male', 'mdd_first_dx_since',
       'mdd_latest_dx_since', 'total_mdd_episodes', 'SRI_Refills',
       'SNRI_Refills', 'Atypical_Antipsychotic_Refills',
       'Anti_epileptic_agent_Refills', 'Aminoketone_Refills',
       'Benzodiazepine_Refills', 'Monoamine_Oxidase_Type_B_Inhibitor_Refills',
       'Phenothiazine_Refills', 'Tricyclic_Antidepressant_Refills',
       'total_logs', 'avg_energy_score', 'median_energy_score',
       'min_energy_score', 'max_energy_score', 'avg_pleasantness_score',
       'median_pleasantness_score', 'min_pleasantness_score',
       'max_pleasantness_score'],
      dtype='object')

In [11]:
# Remove missing data

# Drop rows
threshold = len(df_2.columns) * 0.4
# Drop rows with less than the threshold of non-missing values
df_cleaned = df_2.dropna(thresh=threshold)
#df_cleaned.shape

df_2 = drop_missing_columns(df_cleaned, threshold=0.8)

print(df_2.shape)

(313, 24)


In [12]:
#standardized_df,scaler = standardize_dataframe_sklearn(df_2)

#original_df = unstandardize_dataframe(standardized_df, scaler)

In [13]:
#standardized_df.columns

In [14]:
df_2.columns

Index(['pat_age', 'Is_Female', 'Is_Male', 'mdd_first_dx_since',
       'mdd_latest_dx_since', 'total_mdd_episodes', 'SRI_Refills',
       'SNRI_Refills', 'Atypical_Antipsychotic_Refills',
       'Anti_epileptic_agent_Refills', 'Aminoketone_Refills',
       'Benzodiazepine_Refills', 'Monoamine_Oxidase_Type_B_Inhibitor_Refills',
       'Phenothiazine_Refills', 'Tricyclic_Antidepressant_Refills',
       'total_logs', 'avg_energy_score', 'median_energy_score',
       'min_energy_score', 'max_energy_score', 'avg_pleasantness_score',
       'median_pleasantness_score', 'min_pleasantness_score',
       'max_pleasantness_score'],
      dtype='object')

In [15]:
df_2_med_profile = df_2[['pat_age','Is_Female', 'Is_Male', 'mdd_first_dx_since',
       'mdd_latest_dx_since', 'total_mdd_episodes', 'SRI_Refills',
       'SNRI_Refills', 'Atypical_Antipsychotic_Refills',
       'Anti_epileptic_agent_Refills', 'Aminoketone_Refills',
       'Benzodiazepine_Refills', 'Monoamine_Oxidase_Type_B_Inhibitor_Refills',
       'Phenothiazine_Refills', 'Tricyclic_Antidepressant_Refills', 
        'avg_energy_score','median_energy_score','min_energy_score','max_energy_score',
        'avg_pleasantness_score','median_pleasantness_score','min_pleasantness_score',
        'max_pleasantness_score'
        ]]

In [16]:
demo_features_list = ['pat_age','Is_Female', 'Is_Male', 'mdd_first_dx_since',
       'mdd_latest_dx_since', 'total_mdd_episodes', 'SRI_Refills',
       'SNRI_Refills', 'Atypical_Antipsychotic_Refills',
       'Anti_epileptic_agent_Refills', 'Aminoketone_Refills',
       'Benzodiazepine_Refills', 'Monoamine_Oxidase_Type_B_Inhibitor_Refills',
       'Phenothiazine_Refills', 'Tricyclic_Antidepressant_Refills']

behavior_features_list = ['avg_energy_score','median_energy_score','min_energy_score','max_energy_score',
        'avg_pleasantness_score','median_pleasantness_score','min_pleasantness_score',
        'max_pleasantness_score']

In [17]:
df_3_med_profile = df_2_med_profile.fillna(0)
#df_3_med_profile[behavior_features_list] = df_2_med_profile[behavior_features_list]
#nan_rows = df_4[df_4.isnull().any(axis=1)]
#print(nan_rows)


In [18]:
df_3_med_profile_stand,scaler = standardize_dataframe_sklearn(df_3_med_profile)
dbscan = DBSCAN(eps=0.7, min_samples=2)
clusters = dbscan.fit_predict(df_3_med_profile_stand)

df_3_med_profile['cluster'] = clusters
df_3_med_profile.groupby('cluster').size()
# Add cluster labels to DataFrame


cluster
-1     237
 0       2
 1      49
 2       2
 3       3
 4       2
 5       2
 6       2
 7       2
 8       2
 9       3
 10      3
 11      2
 12      2
dtype: int64

In [19]:
for i in np.arange(1.0,0.5,-0.1): 
    #print(i)
    print('--------------------------------')
    for j in range(10,4,-1): 
        print('eps: %f' %i)
        print('min sample: %d' %j)
        dbscan = DBSCAN(eps=i, min_samples=j)
        clusters = dbscan.fit_predict(df_3_med_profile_stand)
        df_3_med_profile['cluster'] = clusters
        print(df_3_med_profile.groupby('cluster').size())

--------------------------------
eps: 1.000000
min sample: 10
cluster
-1    251
 0     62
dtype: int64
eps: 1.000000
min sample: 9
cluster
-1    249
 0     64
dtype: int64
eps: 1.000000
min sample: 8
cluster
-1    248
 0     65
dtype: int64
eps: 1.000000
min sample: 7
cluster
-1    248
 0     65
dtype: int64
eps: 1.000000
min sample: 6
cluster
-1    231
 0     70
 1      6
 2      6
dtype: int64
eps: 1.000000
min sample: 5
cluster
-1    227
 0     72
 1      8
 2      6
dtype: int64
--------------------------------
eps: 0.900000
min sample: 10
cluster
-1    274
 0     39
dtype: int64
eps: 0.900000
min sample: 9
cluster
-1    263
 0     44
 1      6
dtype: int64
eps: 0.900000
min sample: 8
cluster
-1    261
 0     52
dtype: int64
eps: 0.900000
min sample: 7
cluster
-1    257
 0     56
dtype: int64
eps: 0.900000
min sample: 6
cluster
-1    257
 0     56
dtype: int64
eps: 0.900000
min sample: 5
cluster
-1    248
 0     59
 1      6
dtype: int64
--------------------------------
eps: 0.8000

In [20]:
###
# Looks like Eps = 1.0 & min_sample = 6 could be appropriate size. 

df_3_med_profile_stand,scaler = standardize_dataframe_sklearn(df_3_med_profile)
#dbscan = DBSCAN(eps=0.8, min_samples=4)
dbscan = DBSCAN(eps=1.0, min_samples=6)
clusters = dbscan.fit_predict(df_3_med_profile_stand)

df_3_med_profile['cluster'] = clusters
df_3_med_profile.groupby('cluster').size()

cluster
-1    244
 0     20
 1     14
 2     23
 3      6
 4      6
dtype: int64

In [22]:
clusters_df = df_3_med_profile.groupby('cluster').agg(['mean', 'std', 'median', 'min', 'max'])

In [23]:
clusters_df.columns

MultiIndex([(               'pat_age',   'mean'),
            (               'pat_age',    'std'),
            (               'pat_age', 'median'),
            (               'pat_age',    'min'),
            (               'pat_age',    'max'),
            (             'Is_Female',   'mean'),
            (             'Is_Female',    'std'),
            (             'Is_Female', 'median'),
            (             'Is_Female',    'min'),
            (             'Is_Female',    'max'),
            ...
            ('min_pleasantness_score',   'mean'),
            ('min_pleasantness_score',    'std'),
            ('min_pleasantness_score', 'median'),
            ('min_pleasantness_score',    'min'),
            ('min_pleasantness_score',    'max'),
            ('max_pleasantness_score',   'mean'),
            ('max_pleasantness_score',    'std'),
            ('max_pleasantness_score', 'median'),
            ('max_pleasantness_score',    'min'),
            ('max_pleasantness_sco

In [26]:
cured_patients = df_3_med_profile.loc[df_3_med_profile['cluster'] == 1,['cluster']]

In [27]:
type(cured_patients)

pandas.core.frame.DataFrame

In [28]:
pd.set_option('display.max_columns', None)

In [29]:
clusters_df[[('pat_age', 'mean'),('pat_age', 'median'),('Is_Female', 'mean'),('Is_Female', 'median')
             ,('Is_Male', 'mean'),('Is_Male', 'median')
             ,('mdd_first_dx_since', 'mean'),('mdd_first_dx_since', 'median'),('mdd_latest_dx_since', 'mean'), ('mdd_latest_dx_since', 'median')
             ,('total_mdd_episodes', 'mean'),('total_mdd_episodes', 'median'),('SRI_Refills', 'mean'), ('SRI_Refills', 'median')
            ,('Atypical_Antipsychotic_Refills', 'mean'),('Atypical_Antipsychotic_Refills', 'median')
            ,('Anti_epileptic_agent_Refills', 'mean'),('Anti_epileptic_agent_Refills', 'median')
             ,('Aminoketone_Refills', 'mean'),('Aminoketone_Refills', 'median'),('Benzodiazepine_Refills', 'mean')
            ,('Benzodiazepine_Refills', 'median'),('Monoamine_Oxidase_Type_B_Inhibitor_Refills', 'mean')
            ,('Monoamine_Oxidase_Type_B_Inhibitor_Refills', 'median'),('Phenothiazine_Refills', 'mean'),('Phenothiazine_Refills', 'median')
            ,('Tricyclic_Antidepressant_Refills', 'mean'),('Tricyclic_Antidepressant_Refills', 'median')
            ]]

pat_age        Is_Female          Is_Male         \
              mean median      mean median      mean median   
cluster                                                       
-1       39.434426   37.0  0.737705    1.0  0.262295    0.0   
 0       29.600000   30.0  1.000000    1.0  0.000000    0.0   
 1       31.642857   32.0  1.000000    1.0  0.000000    0.0   
 2       45.304348   45.0  1.000000    1.0  0.000000    0.0   
 3       50.833333   50.0  0.000000    0.0  1.000000    1.0   
 4       31.833333   33.5  0.000000    0.0  1.000000    1.0   

        mdd_first_dx_since         mdd_latest_dx_since          \
                      mean  median                mean  median   
cluster                                                          
-1             1763.168033  1417.5          440.077869   235.0   
 0             1129.900000  1129.5          968.250000   974.0   
 1             1764.642857  1883.0          403.071429   408.0   
 2             1885.652174  1846.0         1419.913043  1359.0   
 3              759.166667   671.0          410.333333   390.0   
 4              887.333333   871.0          865.333333   824.5   

        total_mdd_episodes        SRI_Refills         \
                      mean median        mean median   
cluster                                                
-1                6.372951    5.0    7.750000    2.0   
 0                1.850000    2.0    0.800000    1.0   
 1                4.785714    5.0    2.714286    1.5   
 2                3.434783    3.0    0.913043    0.0   
 3                2.333333    2.0    0.500000    0.0   
 4                1.500000    1.5    1.000000    0.0   

        Atypical_Antipsychotic_Refills        Anti_epileptic_agent_Refills  \
                                  mean median                         mean   
cluster                                                                      
-1                            5.721311    0.0                     3.393443   
 0                            0.600000    0.0                     0.250000   
 1                            0.142857    0.0                     0.642857   
 2                            0.608696    0.0                     0.260870   
 3                            0.000000    0.0                     0.000000   
 4                            1.166667    1.0                     0.000000   

               Aminoketone_Refills        Benzodiazepine_Refills         \
        median                mean median                   mean median   
cluster                                                                   
-1         0.0            2.098361    0.0               1.331967    0.0   
 0         0.0            0.150000    0.0               0.050000    0.0   
 1         0.0            0.571429    0.0               0.071429    0.0   
 2         0.0            0.782609    0.0               0.043478    0.0   
 3         0.0            0.833333    0.5               0.000000    0.0   
 4         0.0            0.166667    0.0               0.333333    0.0   

        Monoamine_Oxidase_Type_B_Inhibitor_Refills         \
                                              mean median   
cluster                                                     
-1                                        0.036885    0.0   
 0                                        0.000000    0.0   
 1                                        0.000000    0.0   
 2                                        0.000000    0.0   
 3                                        0.000000    0.0   
 4                                        0.000000    0.0   

        Phenothiazine_Refills        Tricyclic_Antidepressant_Refills         
                         mean median                             mean median  
cluster                                                                       
-1                   0.012295    0.0                         0.893443    0.0  
 0                   0.000000    0.0                         0.050000    0.0  
 1                 

## Cluster 1: 
### Insights:
#### 1. Patients with less number of episodes. And last episode happened about ~1000 days back. 
#### 2. 

In [30]:
pd.reset_option('display.max_columns')

In [31]:
#index_solved = unstandardized_df_3_med_profile.loc[unstandardized_df_3_med_profile['cluster'] == 1].index.get_level_values(0)

In [32]:
#index_series = pd.Series(index_solved)
#index_series.to_csv("solved_pats.csv", index=False)

In [35]:
df_4_med_profile = df_3_med_profile[df_3_med_profile['cluster'] == -1]

In [37]:
### The long time sufferers clusters
df_4_med_profile_stand,scaler = standardize_dataframe_sklearn(df_4_med_profile)
for i in np.arange(1.0,0.5,-0.1): 
    #print(i)
    print('--------------------------------')
    for j in range(6,1,-1): 
        print('eps: %f' %i)
        print('min sample: %d' %j)
        dbscan = DBSCAN(eps=i, min_samples=j)
        clusters = dbscan.fit_predict(df_4_med_profile_stand)
        #print(len(clusters))
        
        df_4_med_profile['cluster'] = clusters
        #print(len(unstandardized_df_4_med_profile))
        print(df_4_med_profile.groupby('cluster').size())

--------------------------------
eps: 1.000000
min sample: 6
cluster
-1    238
 0      6
dtype: int64
eps: 1.000000
min sample: 5
cluster
-1    233
 0      6
 1      5
dtype: int64
eps: 1.000000
min sample: 4
cluster
-1    226
 0      4
 1      6
 2      8
dtype: int64
eps: 1.000000
min sample: 3
cluster
-1    216
 0      9
 1      4
 2      3
 3      6
 4      3
 5      3
dtype: int64
eps: 1.000000
min sample: 2
cluster
-1     194
 0       2
 1       9
 2       2
 3       4
 4       2
 5       3
 6       6
 7       2
 8       2
 9       2
 10      2
 11      2
 12      3
 13      2
 14      2
 15      3
 16      2
dtype: int64
--------------------------------
eps: 0.900000
min sample: 6
cluster
-1    244
dtype: int64
eps: 0.900000
min sample: 5
cluster
-1    244
dtype: int64
eps: 0.900000
min sample: 4
cluster
-1    234
 0      4
 1      6
dtype: int64
eps: 0.900000
min sample: 3
cluster
-1    222
 0      6
 1      4
 2      3
 3      3
 4      3
 5      3
dtype: int64
eps: 0.900000
m

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_2745/1686280247.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_4_med_profile['cluster'] = clusters
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_2745/1686280247.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_4_med_profile['cluster'] = clusters
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_2745/1686280247.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

In [ ]:
## Lets do a K-Means clustering

In [38]:
#unstandardized_df_5_med_profile = unstandardized_df_4_med_profile[unstandardized_df_4_med_profile['cluster'] == -1]
#unstandardized_df_5_med_profile = unstandardized_df_5_med_profile.iloc[:,:-1]

In [39]:
df_4_med_profile.shape

(244, 24)

In [40]:
df_4_med_profile_stand,scaler = standardize_dataframe_sklearn(df_4_med_profile)
kmeans = KMeans(n_clusters=4, random_state=42)
df_4_med_profile_stand['cluster'] = kmeans.fit_predict(df_4_med_profile_stand)

In [41]:
print(df_4_med_profile_stand.groupby('cluster').size())

cluster
0     82
1    113
2      6
3     43
dtype: int64


In [42]:
df_4_med_profile['cluster'] = df_4_med_profile_stand['cluster']

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_2745/3312502011.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_4_med_profile['cluster'] = df_4_med_profile_stand['cluster']


In [43]:
clusters_df = df_4_med_profile.groupby('cluster').agg(['mean', 'std', 'median', 'min', 'max'])

In [45]:
print(df_4_med_profile.groupby('cluster').size())

cluster
0     82
1    113
2      6
3     43
dtype: int64


In [46]:
pd.set_option('display.max_columns', None)
clusters_df[[('pat_age', 'mean'),('pat_age', 'median'),('Is_Female', 'mean'),('Is_Female', 'median')
             ,('Is_Male', 'mean'),('Is_Male', 'median')
             ,('mdd_first_dx_since', 'mean'),('mdd_first_dx_since', 'median'),('mdd_latest_dx_since', 'mean'), ('mdd_latest_dx_since', 'median')
             ,('total_mdd_episodes', 'mean'),('total_mdd_episodes', 'median'),('SRI_Refills', 'mean'), ('SRI_Refills', 'median')
            ,('Atypical_Antipsychotic_Refills', 'mean'),('Atypical_Antipsychotic_Refills', 'median')
            ,('Anti_epileptic_agent_Refills', 'mean'),('Anti_epileptic_agent_Refills', 'median')
             ,('Aminoketone_Refills', 'mean'),('Aminoketone_Refills', 'median'),('Benzodiazepine_Refills', 'mean')
            ,('Benzodiazepine_Refills', 'median'),('Monoamine_Oxidase_Type_B_Inhibitor_Refills', 'mean')
            ,('Monoamine_Oxidase_Type_B_Inhibitor_Refills', 'median'),('Phenothiazine_Refills', 'mean'),('Phenothiazine_Refills', 'median')
            ,('Tricyclic_Antidepressant_Refills', 'mean'),('Tricyclic_Antidepressant_Refills', 'median')
            ]]


pat_age        Is_Female          Is_Male         \
              mean median      mean median      mean median   
cluster                                                       
0        40.719512   39.5  1.000000    1.0  0.000000    0.0   
1        40.318584   37.0  0.725664    1.0  0.274336    0.0   
2        46.333333   45.5  0.833333    1.0  0.166667    0.0   
3        33.697674   30.0  0.255814    0.0  0.744186    1.0   

        mdd_first_dx_since         mdd_latest_dx_since         \
                      mean  median                mean median   
cluster                                                         
0              2153.853659  1682.5          470.182927  329.5   
1              1270.902655   950.0          215.362832  161.0   
2              4131.000000  3900.0          189.500000  144.5   
3              1981.372093  1742.0         1008.162791  454.0   

        total_mdd_episodes        SRI_Refills         \
                      mean median        mean median   
cluster                                                
0                 6.317073    5.0    7.780488    3.0   
1                 5.743363    4.0    6.707965    2.0   
2                26.000000   24.5   47.000000   41.0   
3                 5.395349    3.0    4.953488    2.0   

        Atypical_Antipsychotic_Refills        Anti_epileptic_agent_Refills  \
                                  mean median                         mean   
cluster                                                                      
0                             3.731707    1.0                     2.695122   
1                             6.221239    0.0                     2.637168   
2                            43.833333   46.5                    28.500000   
3                             2.883721    0.0                     3.209302   

               Aminoketone_Refills        Benzodiazepine_Refills         \
        median                mean median                   mean median   
cluster                                                                   
0          0.0            1.560976    0.0               0.609756    0.0   
1          0.0            1.938053    0.0               0.831858    0.0   
2         16.0           16.333333    9.0              13.333333    6.0   
3          0.0            1.558140    0.0               2.348837    0.0   

        Monoamine_Oxidase_Type_B_Inhibitor_Refills         \
                                              mean median   
cluster                                                     
0                                              0.0    0.0   
1                                              0.0    0.0   
2                                              1.5    0.0   
3                                              0.0    0.0   

        Phenothiazine_Refills        Tricyclic_Antidepressant_Refills         
                         mean median                             mean median  
cluster                                                                       
0                         0.0    0.0                         1.073171    0.0  
1                         0.0    0.0                         1.079646    0.0  
2                         0.5    0.0                         0.166667    0.0  
3                         0.0    0.0                         0.162791    0.0

In [47]:
pd.reset_option('display.max_columns')

In [50]:
amino_heavy_users = df_4_med_profile.loc[df_4_med_profile['cluster'] == 1,['cluster']]

In [51]:
SRI_Heavy_users = df_4_med_profile.loc[df_4_med_profile['cluster'] == 0,['cluster']]

## Insights of the clusters

In [52]:
df_2.head(5)

,,pat_age,Is_Female,Is_Male,mdd_first_dx_since,mdd_latest_dx_since,total_mdd_episodes,SRI_Refills,SNRI_Refills,Atypical_Antipsychotic_Refills,Anti_epileptic_agent_Refills,...,Tricyclic_Antidepressant_Refills,total_logs,avg_energy_score,median_energy_score,min_energy_score,max_energy_score,avg_pleasantness_score,median_pleasantness_score,min_pleasantness_score,max_pleasantness_score
patient_id,preferred_name,,,,,,,,,,,,,,,,,,,,,
MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,Jesse Burns,31,0,1,556,72,3,1.0,0.0,4.0,18.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MTIzNDU2Nzg5MGFiY2RlZg3CXB2zJYZRiBxVgjM8zednndAblIo2aoGrPHt49HPL1XPuB4A2qHWecNCKJADUQeOM3L5D0CFScSUGmlDWcBVD7YATWARmT23ZW5DM5pcq,Lisa Cooper,79,1,0,330,330,1,9.0,5.0,2.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MTIzNDU2Nzg5MGFiY2RlZg8Ql415Ica3v8I1I5VPuFn5M2fSQG9N9HSdpAPPGCNaNywg837/aw1MEUzX7xxVqEvgR09LizXzW13w3nYQUgD3uIlsLpxcf26VrMvEK9vz,Nicole Baker,24,1,0,138,126,3,0.0,0.0,4.0,0.0,...,0.0,9.0,4.44,5.0,3.33,5.0,6.48,6.67,3.33,8.33
MTIzNDU2Nzg5MGFiY2RlZgbPUqLewgwJaCnqdTUKhaMBRyADgg09cCtOObK5XpfogwShkg0xU7WTPEKA1wXEpndE5JRA/lE559tNdpGzeVVYd96xy2hOew6lv2lx9lyS,Mackenzie Ramos,69,1,0,7936,1106,6,5.0,1.0,0.0,2.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MTIzNDU2Nzg5MGFiY2RlZgC3omK5hr/FNtAbgYBTs3BFkeZoUHbhXgOTOjbnlp2rFyxInMlhMeub4A429irdS/zsw/bufgON173aocygBomGt4LFjD6rmIuA6o9sbQ2L,Monica Torres,23,1,0,976,224,9,12.0,9.0,4.0,8.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
matching_index = df_2.index.intersection(cured_patients.index)
df_2.loc[matching_index,'profile'] = 'less_number_of_episodes & sparsely spaced'

In [54]:
matching_index = df_2.index.intersection(amino_benzo_users.index)
df_2.loc[matching_index,'profile'] = 'amino_heavy_users'

NameError: name 'amino_benzo_users' is not defined

In [ ]:
matching_index = df_2.index.intersection(SRI_Heavy_users.index)
df_2.loc[matching_index,'profile'] = 'SRI_heavy_users'

In [55]:
df_2['profile'] = df_2['profile'].fillna('Others')

In [56]:
df_2

,,pat_age,Is_Female,Is_Male,mdd_first_dx_since,mdd_latest_dx_since,total_mdd_episodes,SRI_Refills,SNRI_Refills,Atypical_Antipsychotic_Refills,Anti_epileptic_agent_Refills,...,total_logs,avg_energy_score,median_energy_score,min_energy_score,max_energy_score,avg_pleasantness_score,median_pleasantness_score,min_pleasantness_score,max_pleasantness_score,profile
patient_id,preferred_name,,,,,,,,,,,,,,,,,,,,,
MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,Jesse Burns,31,0,1,556,72,3,1.0,0.0,4.0,18.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Others
MTIzNDU2Nzg5MGFiY2RlZg3CXB2zJYZRiBxVgjM8zednndAblIo2aoGrPHt49HPL1XPuB4A2qHWecNCKJADUQeOM3L5D0CFScSUGmlDWcBVD7YATWARmT23ZW5DM5pcq,Lisa Cooper,79,1,0,330,330,1,9.0,5.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Others
MTIzNDU2Nzg5MGFiY2RlZg8Ql415Ica3v8I1I5VPuFn5M2fSQG9N9HSdpAPPGCNaNywg837/aw1MEUzX7xxVqEvgR09LizXzW13w3nYQUgD3uIlsLpxcf26VrMvEK9vz,Nicole Baker,24,1,0,138,126,3,0.0,0.0,4.0,0.0,...,9.0,4.44,5.00,3.33,5.00,6.48,6.67,3.33,8.33,Others
MTIzNDU2Nzg5MGFiY2RlZgbPUqLewgwJaCnqdTUKhaMBRyADgg09cCtOObK5XpfogwShkg0xU7WTPEKA1wXEpndE5JRA/lE559tNdpGzeVVYd96xy2hOew6lv2lx9lyS,Mackenzie Ramos,69,1,0,7936,1106,6,5.0,1.0,0.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Others
MTIzNDU2Nzg5MGFiY2RlZgC3omK5hr/FNtAbgYBTs3BFkeZoUHbhXgOTOjbnlp2rFyxInMlhMeub4A429irdS/zsw/bufgON173aocygBomGt4LFjD6rmIuA6o9sbQ2L,Monica Torres,23,1,0,976,224,9,12.0,9.0,4.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Others
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MTIzNDU2Nzg5MGFiY2RlZvkiTMKB2TJpMiIKm0Ke7TMDOCk/nh6UAX4a58C6RA4twDYsqJurrBUUKrgwqxhPXX7wFl3qTDVPbAx7qwe7d6kzNtmo57voqHkN6umD3m5J,Nicole Ramirez,53,0,1,980,23,6,9.0,0.0,6.0,0.0,...,1.0,3.33,3.33,3.33,3.33,3.33,3.33,3.33,3.33,Others
MTIzNDU2Nzg5MGFiY2RlZvmQNDpZTrSBry0hMOsCB1wKBrgwuCFnKz/zMhyvBCVLA0Nf1Vmk8O39RrSmL+IZkWnMbkCXBhkw4s6y0YMG/XJUYxKeOrVwdnyCY8GiJmPc,Barbara Stevenson,40,0,1,1996,145,3,9.0,0.0,17.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Others
MTIzNDU2Nzg5MGFiY2RlZvMt5XPW1ev1t/DurAB1I9Doj1NFJo13ISbtuVYFdwYa2yqaNZCRT/NaVTv0mLpU1exkDhbi9aiwplckE+OoSITWDEY1uPWmBP9XivmFPFZ3,Sabrina Bowen,37,1,0,1982,719,10,0.0,4.0,7.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Others


In [57]:
print(df_2.groupby('profile').size())

profile
Others                                       299
less_number_of_episodes & sparsely spaced     14
dtype: int64


In [58]:
clusters_df = df_2.groupby('profile').agg(['mean', 'std', 'median', 'min', 'max'])

In [59]:
pd.set_option('display.max_columns', None)

In [60]:
clusters_df

pat_age                        \
                                                mean        std median min   
profile                                                                      
Others                                     39.304348  14.667604   37.0  12   
less_number_of_episodes & sparsely spaced  31.642857   5.513719   32.0  20   

                                              Is_Female                      \
                                          max      mean      std median min   
profile                                                                       
Others                                     82  0.745819  0.43613    1.0   0   
less_number_of_episodes & sparsely spaced  40  1.000000  0.00000    1.0   1   

                                                Is_Male                      \
                                          max      mean      std median min   
profile                                                                       
Others                                      1  0.254181  0.43613    0.0   0   
less_number_of_episodes & sparsely spaced   1  0.000000  0.00000    0.0   0   

                                              mdd_first_dx_since               \
                                          max               mean          std   
profile                                                                         
Others                                      1        1692.508361  1497.608466   
less_number_of_episodes & sparsely spaced   0        1764.642857   700.895867   

                                                              \
                                           median  min   max   
profile                                                        
Others                                     1351.0   29  8019   
less_number_of_episodes & sparsely spaced  1883.0  582  3149   

                                          mdd_latest_dx_since              \
                                                         mean         std   
profile                                                                     
Others                                             558.715719  643.862258   
less_number_of_episodes & sparsely spaced          403.071429  191.784119   

                                                             \
                                          median  min   max   
profile                                                       
Others                                     332.0   -6  5149   
less_number_of_episodes & sparsely spaced  408.0  125   787   

                                          total_mdd_episodes                   \
                                                        mean       std median   
profile                                                                         
Others                                              5.665552  5.696624    4.0   
less_number_of_episodes & sparsely spaced           4.785714  1.717716    5.0   

                                                  SRI_Refills             \
                                          min max        mean        std   
profile                                                                    
Others                                      1  42    6.917857  12.774458   
less_number_of_episodes & sparsely spaced   2   8    2.714286   3.625308   

                                                            SNRI_Refills  \
                                          median  min   max         mean   
profile                                                                    
Others                                       2.0  0.0  78.0     4.217857   
less_number_of_episodes & sparsely spaced    1.5  0.0  11.0     1.285714   

                                                                        \
                                                 std median  min   max   
profile                                                                  
Others                                     

In [ ]:
pd.reset_option('display.max_columns')

In [ ]:
print(df_2.groupby(['Is_Female','profile']).size())

In [ ]:
print(df_2.groupby(['Is_Male','profile']).size())